# Family-Ladder Scaling Induction Study

Every prior induction study (`periodic`, `chromatic`, `periodic_moe`,
`periodic_coprime`, `periodic_divisor`) held the **model** fixed (one small
trio) and varied the **quiz** (sequence length, rule count, noise). This
study inverts that: the quiz is the plain `periodic_moe` baseline
(`PeriodicConfig(n=9, labels=9)`, unmodified), held fixed, and what varies is
the **model** -- 7 vendor families x 3 rungs each (smallest / geometric-
middle / largest checkpoint on that family's public ladder) = 21 checkpoints.
Holding the quiz fixed is what makes an accuracy difference between two
rungs of one family, or between two families at a comparable rung,
attributable to parameter count rather than to a changed task.

All study config (seeds, info arms, per-model CoT toggles, prompt template,
budget derivation) lives in `run_study.py`, the single source of truth this
notebook imports rather than re-declares -- see that file's module
docstring for the full rationale.

## Roster (21 checkpoints)



Spec keys and analysis tags are `run_study.MODELS`; instance tiers are
`smolbench.evals.ec2.EC2_DEPLOY_SPECS`'s "Family-ladder scaling study
roster" comment (tier A = g6e.4xlarge, 1x L40S; tier B = g6e.12xlarge, 4x
L40S; tier C = p5.48xlarge, 8x H100; tier D = p5e.48xlarge, 8x H200).

| Family | Rung | Spec key | Analysis tag | Instance tier |
|---|---|---|---|---|
| Qwen3.5 (Alibaba, CN) | 27B | `qwen3.5-27b` | `qwen35_27b` | B (g6e.12xlarge) |
| Qwen3.5 (Alibaba, CN) | 122B-A10B | `qwen3.5-122b-a10b` | `qwen35_122b` | C (p5.48xlarge) |
| Qwen3.5 (Alibaba, CN) | 397B-A17B (FP8) | `qwen3.5-397b-a17b` | `qwen35_397b` | C (p5.48xlarge) |
| Nemotron-3 (NVIDIA, US) | Nano-4B | `nemotron-3-nano-4b` | `nemo3_4b` | A (g6e.4xlarge) |
| Nemotron-3 (NVIDIA, US) | Nano-30B-A3B | `nemotron-3-nano-30b-a3b` | `nemo3_30b` | B (g6e.12xlarge) |
| Nemotron-3 (NVIDIA, US) | Super-120B-A12B | `nemotron-3-super-120b-a12b` | `nemo3_120b` | C (p5.48xlarge) |
| Gemma-4 (Google, US) | E2B | `gemma-4-e2b` | `gemma4_e2b` | A (g6e.4xlarge) |
| Gemma-4 (Google, US) | 12B | `gemma-4-12b` | `gemma4_12b` | A (g6e.4xlarge) |
| Gemma-4 (Google, US) | 31B | `gemma-4-31b` | `gemma4_31b` | B (g6e.12xlarge) |
| GLM-4.x (Zhipu/Z.ai, CN) | 4.7-Flash | `glm-4.7-flash` | `glm_flash` | B (g6e.12xlarge) |
| GLM-4.x (Zhipu/Z.ai, CN) | 4.5-Air | `glm-4.5-air` | `glm_air` | C (p5.48xlarge) |
| GLM-4.x (Zhipu/Z.ai, CN) | 4.7 (cross-generation) | `glm-4.7` | `glm_47` | D (p5e.48xlarge) |
| Ministral-3 (Mistral, FR) | 3B | `ministral-3-3b` | `min3_3b` | A (g6e.4xlarge) |
| Ministral-3 (Mistral, FR) | 8B | `ministral-3-8b` | `min3_8b` | B (g6e.12xlarge) |
| Ministral-3 (Mistral, FR) | 14B | `ministral-3-14b` | `min3_14b` | B (g6e.12xlarge) |
| EXAONE (LG AI Research, KR) | 4.0-32B | `exaone-4.0-32b` | `exaone_32b` | B (g6e.12xlarge) |
| EXAONE (LG AI Research, KR) | 4.5-33B | `exaone-4.5-33b` | `exaone_33b` | B (g6e.12xlarge) |
| EXAONE (LG AI Research, KR) | K-EXAONE-236B-A23B (cross-gen) | `k-exaone-236b-a23b` | `exaone_236b` | C (p5.48xlarge) |
| DeepSeek (CN) | V4-Flash | `deepseek-v4-flash` | `ds_flash` | C (p5.48xlarge) |
| DeepSeek (CN) | V3.1 | `deepseek-v3.1` | `ds_v31` | D (p5e.48xlarge) |
| DeepSeek (CN) | V4-Pro (cross-generation) | `deepseek-v4-pro` | `ds_pro` | D (p5e.48xlarge) |

**As designed, not as served.** The tier column is what `EC2_DEPLOY_SPECS` requests; spot capacity forced substitutions mid-study, and under the earliest-wins selection rule three lanes are era-split inside their own 30 seeds (`gemma-4-12b` 14+16, `ministral-3-14b` 9+14+7, `deepseek-v4-flash` 12+18). The mixing was measured and priced as noise, not bias (record: `CONFOUND_AUDIT_2026-08-13.md`, archived 2026-08-25).

## Seeds and replicate count

`BASE_SEED = 0`, `n_replicates = 30` (seeds `0..29`) -- **user-locked**, and
deliberately different from every prior induction study's `1776`. It exists
so this study's replicate seeds can never silently alias a sibling study's
even if a results prefix were ever shared by accident.

## Reasoning

CoT is **ON for all 21 checkpoints**. The per-vendor toggle -- which
`chat_template_kwargs` key turns thinking on, and for which checkpoints the
explicit `True` is load-bearing rather than redundant -- lives in
`run_study.COT_ARGS`, a table total over `MODELS` (see that file's module
docstring, "Reasoning: CoT is ON for every model in this study").

## Cost warning

This study provisions up to **21 concurrent EC2 spot instances**, spanning
`g6e.4xlarge` (1x L40S) to `p5e.48xlarge` (8x H200), each billed for the
duration it is up. **Nothing in this notebook provisions anything.** Unlike
every sibling induction notebook (`periodic`, `chromatic`, `periodic_moe`,
...), which each provision and run from cells with a `provision()` /
`run()` / `teardown()` sequence, this study's fleet is launched from a
**terminal**, outside any kernel session -- see Section 5 below. Muscle
memory from those sibling notebooks is to run-all -- so, to be unambiguous:
running this notebook top-to-bottom does **not** launch, and cannot
accidentally launch, any of the 21 boxes.


In [ ]:
import logging
import sys
from pathlib import Path

from dotenv import load_dotenv

# CRITICAL, import-order trap: smolbench.evals.ec2 freezes its EC2_* module
# constants (EC2_EXPERIMENT_TAG, EC2_INSTANCE_TYPES, ...) from os.environ at
# IMPORT time, not at call time (see that module's docstring, "Env-read
# timing"). load_dotenv MUST therefore run before the first `smolbench`
# import anywhere in this kernel, or this study's config is silently frozen
# to un-overridden defaults for the life of the kernel -- with no error to
# signal it.

# Resolve the notebook's own directory without relying on cwd. The sibling
# periodic_moe notebook's cell 1 does `load_dotenv(Path.cwd() / "keys.env")`
# unguarded -- a kernel started from the repo root, or from any other cwd,
# makes that silently load nothing (load_dotenv returns quietly on a missing
# path) and every EC2_* constant below freezes to its default. We do NOT
# copy that pattern: walk up from cwd looking for the repo root instead.
NB_DIR = Path.cwd() if (Path.cwd() / "keys.env").exists() else None
if NB_DIR is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "smolbench").is_dir():
            NB_DIR = candidate / "notebooks" / "induction"
            break

assert NB_DIR is not None and (NB_DIR / "keys.env").exists(), (
    "could not locate notebooks/induction/keys.env by walking up from this "
    f"kernel's cwd ({Path.cwd()}) to a directory containing both "
    "pyproject.toml and smolbench/. Start the kernel from inside the repo "
    "(or a subdirectory of it)."
)
# REPO_ROOT anchors the `scripts` namespace-package import in the fleet
# status section below -- it is NB_DIR's grandparent (notebooks/induction ->
# notebooks -> repo root).
REPO_ROOT = NB_DIR.parent.parent

load_dotenv(NB_DIR / "keys.env", verbose=True)
logging.basicConfig(level=logging.INFO)

# run_study.py ALSO calls load_dotenv on this same keys.env file, at its own
# module scope, before its first smolbench import (see its module
# docstring). This cell's call is therefore belt-and-braces, not strictly
# load-bearing by itself -- it exists so the notebook stays correct even if
# a reader inserts a `smolbench` (or `run_study`) import ABOVE this cell,
# which would otherwise freeze the EC2_* constants before either
# load_dotenv call has a chance to run.
sys.path.insert(0, str(NB_DIR))
import run_study


In [ ]:
# Importing rather than re-declaring is what stops this notebook and a
# headless fleet run from drifting apart. Each sibling study's notebook
# carried its own hand-copied prompt template and BASE_SEED constant, and
# that duplication is exactly how two nominally "identical" runs stop being
# identical -- a template edit or a seed change applied to run_study.py
# (the file the fleet actually executes) would otherwise leave this
# notebook silently validating stale prompts.
from run_study import (BASE_SEED, COT_ARGS, EXPERIMENT, INFO_TYPES, MODELS,
                       completion_budget, make_quizzes, template)

print(f"{len(MODELS)} models x {len(INFO_TYPES)} arms x {EXPERIMENT.n_replicates} replicates "
      f"(seeds {EXPERIMENT.seeds[0]}..{EXPERIMENT.seeds[-1]})")


## Prompt Validation


In [ ]:
# Smallest rung on the roster (tier A, single L40S) -- validating against
# ONE model's tokenizer is enough because intens/extens/zero are
# byte-identical across all 21 checkpoints; only noise_intens depends on the
# model argument (see make_quizzes' docstring in run_study.py). Downloads
# ONE tokenizer from HuggingFace and touches no AWS and no GPU.
VALIDATION_MODEL = "nemotron-3-nano-4b"
quizzes = make_quizzes(BASE_SEED, VALIDATION_MODEL)


In [ ]:
print(quizzes["intens"][0].prompt)


In [ ]:
print(quizzes["extens"][0].prompt)


In [ ]:
print(quizzes["zero"][0].prompt)


In [ ]:
from smolbench.evals.tokenization import for_model

# The invariant is PER QUESTION and EXACT -- not on average, not within a
# tolerance -- see tests/induction/test_noise_token_match.py,
# test_periodic_noise_prompt_matches_extens_token_count. noise_intens is a
# LENGTH control: if it were not exactly as long as extens in tokens, an
# intens-vs-extens accuracy gap could be explained by prompt length instead
# of information content, and the arm would be measuring nothing.
tok = for_model(VALIDATION_MODEL)
for extens_q, noise_q in zip(quizzes["extens"], quizzes["noise_intens"]):
    assert tok.count(noise_q.prompt) == tok.count(extens_q.prompt)

# Print both counts so the reader can see the length control actually
# bracket the two arms it sits between.
print(f"noise_intens matched at {tok.count(quizzes['noise_intens'][0].prompt):,} tokens "
      f"(intens is {tok.count(quizzes['intens'][0].prompt):,} tokens)")

# Token-matched is not the same as inert. On this roster the whitespace padding
# destroys the output contract in 6 of 21 models (exaone_32b/33b, min3_8b/14b,
# glm_flash, glm_air -- up to 99.6% non-compliant marks, accuracy 0.000), so a
# low noise_intens score can be an output-collapse artifact rather than an
# information effect. Per the 2026-08-21 user ruling that collapse is a
# first-class result and is never quarantined; significance_report.py prints
# the per-lane collapse census (record: PAIRED_ANALYSIS_RESULTS.md, archived).


In [ ]:
# Same pre-flight arithmetic the fleet runs, per model, before provisioning
# anything -- pure CPU + a HuggingFace tokenizer fetch, no AWS.
print(completion_budget(VALIDATION_MODEL, range(BASE_SEED, BASE_SEED + EXPERIMENT.n_replicates)))


## Fleet Launch

Unlike every sibling induction notebook, this study does **not** run models
from cells: 21 lanes at up to 36 hours each cannot live in one kernel
session, and a dropped kernel would orphan billing instances with nothing
left to tear them down. The fleet is launched from a **terminal**, at the
repo root:

```
nohup .venv/bin/python scripts/fleet/run_fleet.py --phase induction > fleet.out 2>&1 &
```

Useful flags:

- `--phase {induction,deduction,both}` -- which phase(s) to run per lane.
- `--lanes <comma-separated spec keys>` -- restrict to a subset of the 21
  models instead of the full roster.
- `--no-gate` -- skip the family gate described below (only for a lane
  already known-good).
- `--dry-run` -- print what would be launched without provisioning
  anything. **Run this first.**

### The family gate

**Config epoch.** Since 2026-08-18 every `EC2_DEPLOY_SPECS` entry serves the certified determinism bundle (prefix caching off, `--max-num-seqs 1`, `--enforce-eager`, `--seed 0`) plus digest/revision pins. Cross-config agreement was measured at 0/8 prompts, so anything this fleet collects now is config-incomparable with the 2026-08-16 study numbers -- a re-run is a new study, not more of the old one. (Record: `CONTAMINATION_INVENTORY_2026-08-15.md`, archived 2026-08-25 -- see `notebooks/README.md`.)

Rather than gamble on all 21 checkpoints against an image never proven on
the full roster (the digest-pinned `FLEET_IMAGE` since the 2026-08-18
determinism pin; `:nightly` during the study), the supervisor launches tier D
first, then the three cheapest tier-A models (`gemma-4-e2b`,
`nemotron-3-nano-4b`, `ministral-3-3b`). Tiers B and C are held until all
three tier-A lanes report a healthy serve -- turning a 21-way gamble on an
unproven image into a cheap 3-way one before the expensive tiers commit any
spend.

Lane logs land under `notebooks/induction/results/fleet_logs/<spec-key>.log`.


In [ ]:
# scripts/ has no __init__.py -- it imports as a namespace package once the
# repo root is on sys.path, which REPO_ROOT (derived above, in the keys.env
# cell) provides.
sys.path.insert(0, str(REPO_ROOT))          # REPO_ROOT is derived in the keys.env cell above
from scripts.fleet.fleet_status import fleet_rows, format_fleet_table

# READ-ONLY: fleet_rows() is a describe_instances call, server-side filtered
# on tag:smolbench:experiment = scaling-*, so it costs nothing and cannot
# start or stop anything. Needs AWS credentials; returns an explicit
# "no instances" line rather than blank output when the fleet is down, so a
# torn-down fleet is distinguishable from a credentials problem at a glance.
print(format_fleet_table(fleet_rows()))


## Analysis

**Results analysis runs on remote compute, not this host.** The cells below
are written out so the analysis is reproducible and reviewable from this
notebook, but they are gated behind an explicit opt-in so that a casual
run-all on a laptop does not pull the whole S3 results log down.

**Selection rule: earliest wins.** `sync_down()` and `load_marks` resolve the lexicographic *minimum* `run_ts` per (model, seed, arm) -- see `smolbench/evals/results_store.py`'s module docstring. Where a cell was collected twice the analysis reads the pre-re-collection attempt; forced re-runs stay in the S3 log but are invisible to analysis, and voiding data requires explicit exclusion, not a newer object. The published headline on this leg is 119 of the 210 primary contrasts rejected under Holm, on the exact seed-level sign-flip test in `significance_report.py` -- the seed, not the item, is the replicate (records: `PASS_AT_1_REVIEW_PLAN_2026-08-16.md`, `FAMILY_LADDER_ANALYSIS_2026-08-16.md`, archived 2026-08-25 -- see `notebooks/README.md`).


In [ ]:
import os

ANALYSIS_HOST = os.environ.get("ANALYSIS_HOST", "").strip() == "1"
if not ANALYSIS_HOST:
    print("ANALYSIS_HOST != 1 -- skipping. Set ANALYSIS_HOST=1 on the analysis host to run.")
else:
    # sync_down() translates the append-only S3 log back into the local
    # {tag}_{info}/rep_{seed}.yaml layout that power_analysis.py and the
    # figure scripts read. It needs THIS experiment's own model -> tag map
    # (a log key names a model, never a tag), which is why it is called
    # through EXPERIMENT.harness rather than the module-level CLI.
    n = EXPERIMENT.harness.sync_down()
    print(f"synced {n} objects into {EXPERIMENT.results_dir}")
    for model in MODELS:
        # summarize() reads through the results store (S3 requests against
        # an S3-backed store) but can never trigger GPU billing -- see
        # InductionExperiment's module docstring, "Cost warning".
        EXPERIMENT.summarize(model)


In [ ]:
# Power analysis: sizes R from the seed-0 pilot replicate and requires
# This is PROSPECTIVE sizing off the seed-0 pilot under the harmonic-stratified
# CMH -- not achieved power, and NOT this study's primary test. The primary
# inference is the exact seed-level sign-flip in significance_report.py;
# power_analysis.py was never updated for it (it contains no cluster/sign-flip
# code), so its recommended R is not a verdict on the landed data
# (record: POWER_ANALYSIS_2026-08-14.md, archived 2026-08-25).
# sync_down() (above) to have already populated the local results tree.
# Shown as a shell comment, not executed here -- it runs on the analysis
# host, not in this kernel.
#
# uv run --no-project --with numpy --with scipy --with statsmodels \
#     python notebooks/induction/power_analysis.py


## Teardown

No teardown cell here -- teardown is a fleet-lifecycle operation, not a
notebook one:

```
.venv/bin/python scripts/fleet/fleet_teardown.py                 # read-only listing
.venv/bin/python scripts/fleet/fleet_teardown.py --terminate     # actually terminate
```

`run_fleet.py --phase both` shuts each lane's box down itself once the
deduction phase finishes with it, so `fleet_teardown.py` is the **safety
net** -- for `--phase induction`-only runs, lanes that stalled partway, and
anything the supervisor otherwise lost track of.

**Warning:** `run_study.py --teardown` exists but is **standalone-only**
(see its module docstring, "Lifecycle contract") -- it must never be
invoked against a fleet lane, since under the fleet a lane's box is reused
by a later deduction-phase run and a `--teardown` there would terminate the
instance out from under that reattachment.
